# 00 · Carga y ficha del subset

Primer contacto con `subset_ADNC_GNN.h5ad`. Este notebook **no contiene lógica**: llama a las funciones de
`src/tfm/` y enseña los resultados. Todo lo que calcula se guarda en `results/tables/`; el resumen está en `docs/datos.md`.

**Cómo ejecutarlo** (evita el problema del kernel de VS Code, porque la terminal sí activa bien el entorno):

```powershell
conda activate tfm-medal
cd C:\Users\ori\MEDAL\Code
jupyter nbconvert --to notebook --execute --inplace notebooks\00_carga_y_ficha.ipynb
```

Después ábrelo en VS Code para leer las salidas.

In [1]:
import json
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import display

# src/ al path para poder hacer "from tfm import ..." sin instalar nada
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "config.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

from tfm import data
from tfm.config import load_config

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

cfg = load_config(ROOT)
COL = cfg["columns"]
H5AD = cfg["paths"]["subset"]
TABLAS = cfg["paths"]["tables"]
TABLAS.mkdir(parents=True, exist_ok=True)

def guardar(df, nombre):
    df.to_csv(TABLAS / f"{nombre}.csv", encoding="utf-8")
    return df

print("Proyecto:", ROOT)
print("Fichero: ", H5AD, f"({H5AD.stat().st_size / 1024**3:.2f} GB en disco)")
try:
    import psutil
    m = psutil.virtual_memory()
    print(f"RAM total {m.total / 1024**3:.1f} GB, libre {m.available / 1024**3:.1f} GB")
except ImportError:
    pass

Proyecto: C:\Users\ori\MEDAL\Code
Fichero:  C:\Users\ori\MEDAL\Code\Data\raw\subset_ADNC_GNN.h5ad (0.93 GB en disco)
RAM total 15.7 GB, libre 7.6 GB


## 1. Estructura del fichero

Un `.h5ad` guarda un objeto **AnnData**:

| Parte | Qué es | Analogía |
|---|---|---|
| `X` | matriz células × genes (la expresión que se usa por defecto) | matriz de datos N × F |
| `raw` | otra copia de la expresión, normalmente los recuentos originales con todos los genes | la señal antes de preprocesar |
| `layers` | otras versiones de la matriz con los mismos genes que `X` | canales alternativos |
| `obs` | una fila por célula: donante, tipo celular, ADNC... | metadatos de cada muestra |
| `var` | una fila por gen | metadatos de cada característica |
| `obsm` | representaciones por célula de pocas dimensiones (UMAP, scVI) | embeddings |
| `uns` | todo lo demás (parámetros, colores...) | cabecera |

Lo abrimos en **modo backed** (`backed="r"`): `obs` y `var` se cargan en memoria (son pequeños), pero las
matrices se quedan en el disco y solo se leen los trozos que se piden.

In [2]:
print("Matrices en RAM si se cargara todo:", f"{data.estimate_ram_gb(H5AD):.2f} GB")
adata = data.load_adata(H5AD, backed="r")
print(adata)
print("\nraw:", None if adata.raw is None else adata.raw.shape)
print("layers:", [k for k in adata.layers.keys() if k is not None] or "ninguna")
print("obsm:", {k: adata.obsm[k].shape for k in adata.obsm.keys()})
print("uns:", list(adata.uns.keys()))

Matrices en RAM si se cargara todo: 0.91 GB
AnnData object with n_obs × n_vars = 24000 × 6000 backed at 'C:\\Users\\ori\\MEDAL\\Code\\Data\\raw\\subset_ADNC_GNN.h5ad'
    obs: 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'organism_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'is_primary_data', 'Neurotypical reference', 'Class', 'Subclass', 'Supertype', 'Age at death', 'Years of education', 'Cognitive status', 'ADNC', 'Braak stage', 'Thal phase', 'CERAD score', 'APOE4 status', 'Lewy body disease pathology', 'LATE-NC stage', 'Microinfarct pathology', 'Specimen ID', 'donor_id', 'PMI', 'n_counts', 'n_genes', 'Fraction mitochrondrial UMIs', 'suspension_type', 'development_stage_ontology_term_id', 'Continuous Pseudo-progression Score', 'tissue_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'cell

## 2. ¿Recuentos o log-normalizado?

- **Recuentos crudos:** enteros (cuántas moléculas de ARN de cada gen se leyeron en la célula). La suma por
  célula varía mucho porque cada núcleo se secuencia con distinta profundidad.
- **Log-normalizado:** se divide cada célula por su total, se multiplica por una constante `T`
  (típicamente 10.000) y se aplica `log(1 + x)`. Salen decimales, con máximo alrededor de 5–10.

Leemos ~1.000 células de cada matriz directamente del fichero.

In [3]:
mat = guardar(data.describe_matrices(H5AD), "matrices")
display(mat[["formato", "n_filas", "n_cols", "dtype", "densidad", "gb_ram", "min_no_cero", "max",
             "frac_enteros", "suma_celula_mediana", "diagnostico"]])

,formato,n_filas,n_cols,dtype,densidad,gb_ram,min_no_cero,max,frac_enteros,suma_celula_mediana,diagnostico
matriz,,,,,,,,,,,
X,csr_matrix,24000,6000,float32,0.17650,0.189453,0.083273,6.156637,0.0,1174.322998,"log-normalizado (decimales, máx < 20); la suma no es constante porque no est..."
raw/X,csr_matrix,24000,36412,float32,0.11135,0.725084,1.000000,3777.000000,1.0,7483.000000,recuentos crudos (enteros)


### Comprobación fuerte: ¿`X` sale de `raw`?

Si `X = log1p(raw / total_célula × T)`, para cada gen y célula se puede despejar
`T = expm1(X) × total_célula / raw`. Si sale **la misma T en todas las células**, queda demostrado que `X` es
`raw` normalizado, y sabemos a cuánto. También miramos que los ceros coincidan.

In [4]:
norm = data.check_normalization(H5AD)
(TABLAS / "normalizacion.json").write_text(json.dumps(norm, indent=2, ensure_ascii=False), encoding="utf-8")
for k, v in norm.items():
    print(f"{k:30s} {v}")

emparejamiento_genes           X.feature_name = raw.feature_name
genes_X_encontrados_en_raw     6000
genes_X                        6000
frac_ceros_coinciden_X_raw     1.0
celulas_comprobadas            500
T_mediana                      9992.3759765625
T_cv_entre_celulas             0.00043952700798399746
T_cv_dentro_celula_mediana     9.253285782051535e-08
conclusion                     X = log1p(raw / total_celula x 9992), con el total sobre todos los genes de raw


## 3. Columnas de `obs`

Cada columna se clasifica según `configs/config.yaml` (sección `obs_groups`):

- **identificador:** a quién pertenece cada célula.
- **etiqueta:** lo que queremos predecir (ADNC).
- **patologia_prohibida:** componentes o consecuencias del ADNC (Braak, Thal, CERAD, CPS, estado cognitivo...).
  Usarlas como entrada sería **fuga de información**: el modelo leería la respuesta.
- **confusor:** variables que pueden ir de la mano del ADNC y cambiar la expresión (edad, sexo, APOE4, PMI...).
- **tecnica / calidad_celula:** cómo se midió (ensayo, muestra, métricas de QC).
- **tipo_celular:** la anotación en sus distintos niveles.

`nivel` dice si la columna es **de donante** (igual en todas sus células) o **de célula**.

In [5]:
cols_obs = guardar(data.classify_obs_columns(adata.obs, cfg["obs_groups"], COL["donor"]), "obs_columnas")
for g, t in cols_obs.groupby("grupo", observed=True):
    print(f"\n=== {g} ({len(t)}) ===")
    display(t.drop(columns="grupo").set_index("columna"))


=== identificador (2) ===


,dtype,n_valores,n_nulos,nivel,ejemplos
columna,,,,,
donor_id,category,80,0,donante,H21.33.025; H21.33.006; H21.33.021
observation_joinid,object,24000,0,célula,P{?}So$!{<; far&gQ7NjP; ZxjCcK)uO3



=== etiqueta (1) ===


,dtype,n_valores,n_nulos,nivel,ejemplos
columna,,,,,
ADNC,category,4,0,donante,Intermediate; High; Low



=== patologia_prohibida (8) ===


,dtype,n_valores,n_nulos,nivel,ejemplos
columna,,,,,
Braak stage,category,7,0,donante,Braak IV; Braak III; Braak V
CERAD score,category,5,0,donante,Moderate; Sparse; Frequent
Cognitive status,category,3,0,donante,No dementia; Dementia; Reference
Continuous Pseudo-progression Score,float64,76,0,donante,0.5180047374; 0.8380957485; 0.7049965047
Thal phase,category,6,0,donante,Thal 3; Thal 4; Thal 2
condition_celltype,category,24,0,célula,Intermediate_L2IT-GLU; High_L2IT-GLU; Low_L2IT-GLU
disease,category,2,0,donante,normal; dementia
disease_ontology_term_id,category,2,0,donante,PATO:0000461; MONDO:0001627


... (4 outputs left out)


=== calidad_celula (5) ===


,dtype,n_valores,n_nulos,nivel,ejemplos
columna,,,,,
Fraction mitochrondrial UMIs,float32,20352,0,célula,0.005647977; 0.0017333573; 0.016968718
n_counts,int64,15735,0,célula,19476; 61153; 17326
n_genes,int64,7240,0,célula,5044; 8280; 5221
pct_counts_mt,float32,20352,0,célula,0.5647977; 0.17333573; 1.6968718
total_counts,int64,15735,0,célula,19476; 61153; 17326



=== tipo_celular (6) ===


,dtype,n_valores,n_nulos,nivel,ejemplos
columna,,,,,
Class,category,3,0,célula,Neuronal: Glutamatergic; Neuronal: GABAergic; Non-neuronal and Non-neural
Subclass,category,11,0,célula,L2/3 IT; L4 IT; L5 IT
Supertype,category,78,0,célula,L2/3 IT_5; L2/3 IT_1; L4 IT_2
cell_type,category,6,0,célula,L2/3-6 intratelencephalic projecting glutamatergic neuron; sst GABAergic cor...
cell_type_abbr,category,6,0,célula,L2IT-GLU; SST-GABA; VIP-GABA
cell_type_ontology_term_id,category,6,0,célula,CL:4023040; CL:4023017; CL:4023016


## 4. Tabla de donantes

La etiqueta ADNC es **del donante**, no de la célula. Por eso necesitamos una tabla con **una fila por
donante**. `get_donor_table` se queda con las columnas que son constantes dentro de cada donante y **avisa** si
alguna que debería serlo (según `donor_level_expected` en el config) no lo es.

In [6]:
donantes = data.get_donor_table(adata, COL["donor"], expected=cfg["donor_level_expected"])
guardar(donantes, "donantes")
no_const = pd.DataFrame(donantes.attrs["no_constantes"])
guardar(no_const, "donantes_no_constantes")
print(f"{len(donantes)} donantes x {donantes.shape[1]} columnas constantes")
if len(no_const):
    print("\nAVISO: columnas que deberían ser de donante y varían:")
    display(no_const)
else:
    print("Todas las columnas esperadas son constantes dentro de cada donante.")
display(donantes.head())

80 donantes x 29 columnas constantes
Todas las columnas esperadas son constantes dentro de cada donante.


,n_celulas,disease_ontology_term_id,self_reported_ethnicity_ontology_term_id,organism_ontology_term_id,sex_ontology_term_id,tissue_ontology_term_id,Neurotypical reference,Age at death,Years of education,Cognitive status,ADNC,Braak stage,Thal phase,CERAD score,APOE4 status,Lewy body disease pathology,LATE-NC stage,Microinfarct pathology,PMI,suspension_type,development_stage_ontology_term_id,Continuous Pseudo-progression Score,tissue_type,disease,organism,sex,tissue,self_reported_ethnicity,development_stage
donor_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
H18.30.001,441,PATO:0000461,HANCESTRO:0005,NCBITaxon:9606,PATO:0000383,UBERON:0002771,True,Less than 65 years old,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,nucleus,HsapDv:0000154,0.000000,tissue,normal,Homo sapiens,female,middle temporal gyrus,European,60-year-old stage
H18.30.002,1242,PATO:0000461,HANCESTRO:0005,NCBITaxon:9606,PATO:0000384,UBERON:0002771,True,Less than 65 years old,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,nucleus,HsapDv:0000144,0.000000,tissue,normal,Homo sapiens,male,middle temporal gyrus,European,50-year-old stage
H19.30.001,2682,PATO:0000461,HANCESTRO:0005,NCBITaxon:9606,PATO:0000384,UBERON:0002771,True,Less than 65 years old,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,nucleus,HsapDv:0000136,0.000000,tissue,normal,Homo sapiens,male,middle temporal gyrus,European,42-year-old stage
H19.30.002,1492,PATO:0000461,HANCESTRO:0005,NCBITaxon:9606,PATO:0000384,UBERON:0002771,True,Less than 65 years old,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,Reference,nucleus,HsapDv:0000123,0.000000,tissue,normal,Homo sapiens,male,middle temporal gyrus,European,29-year-old stage
H20.33.001,536,PATO:0000461,HANCESTRO:0005,NCBITaxon:9606,PATO:0000384,UBERON:0002771,False,78 to 89 years old,16 to 18 years,No dementia,Low,Braak IV,Thal 2,Sparse,N,Not Identified (olfactory bulb not assessed),LATE Stage 2,0 to 3 microinfarcts,5.9 to 8.7 hours,nucleus,HsapDv:0000208,0.522889,tissue,normal,Homo sapiens,male,middle temporal gyrus,European,82-year-old stage


## 5. Nombres de gen

Problemas típicos de los ficheros de CELLxGENE:

- **Nombres repetidos:** varios IDs de ENSEMBL comparten símbolo. Después hay que decidir qué hacer
  (`var_names_make_unique`, sumar o quitar).
- **IDs de ENSEMBL sin símbolo** (`ENSG000...`): genes sin nombre conocido.
- **Genes mitocondriales** (`MT-`): útiles para el control de calidad.
- **¿Los genes de `X` están todos en `raw`?** En este fichero `raw` usa IDs de ENSEMBL como nombre y guarda el
  símbolo en `feature_name`, mientras que `X` usa símbolos. Para emparejarlos hay que pasar por `feature_name`.

In [7]:
genes, genes_extra = data.check_gene_names(adata)
guardar(genes, "genes_resumen")
(TABLAS / "genes_extra.json").write_text(json.dumps(genes_extra, indent=2, ensure_ascii=False), encoding="utf-8")
display(genes.T)
print(genes_extra)
var_x, var_raw = data.gene_tables(H5AD)
guardar(var_x, "genes_X"); guardar(var_raw, "genes_raw")
print("Emparejamiento X -> raw:", var_x.attrs.get("estrategia"))
nombres_raw = var_raw["feature_name"].astype(str) if "feature_name" in var_raw else var_raw.index.to_series()
rep = nombres_raw[nombres_raw.duplicated(keep=False)]
guardar(rep.value_counts().rename("veces").to_frame(), "genes_repetidos_raw")
print(f"Símbolos repetidos en raw.feature_name: {rep.nunique()}")

matriz,X (índice),X (feature_name),raw (índice),raw (feature_name)
n_genes,6000,6000,36412,36412
n_nombres_repetidos,1,0,0,0
n_filas_con_nombre_repetido,2,0,0,0
ejemplos_repetidos,LINC03021,,,
n_ensembl,953,953,36412,11069
ejemplos_ensembl,ENSG00000176349.11; ENSG00000179066.8; ENSG00000187229.3; ENSG00000188897.10...,ENSG00000176349.11; ENSG00000179066.8; ENSG00000187229.3; ENSG00000188897.10...,ENSG00000000003; ENSG00000000005; ENSG00000000419; ENSG00000000457; ENSG0000...,ENSG00000083622.8; ENSG00000124835.2; ENSG00000131484.4; ENSG00000145063.15;...
n_mitocondriales_MT-,13,13,0,13
n_con_guion_bajo,0,4101,0,18627


{'columnas_var': ['feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'], 'columnas_raw_var': ['feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'], 'emparejamiento_X_raw': 'X.feature_name = raw.feature_name', 'genes_X_no_en_raw': 0, 'ejemplos_X_no_en_raw': [], 'X_feature_biotype': {'gene': 6000}, 'X_feature_is_filtered': {'False': 6000}, 'X_highly_variable': {'True': 6000}}


Emparejamiento X -> raw: X.feature_name = raw.feature_name
Símbolos repetidos en raw.feature_name: 0


## 6. ¿Coincide con la descripción del subset?

24.000 células, 6.000 por clase, 6 tipos celulares, 80 donantes, 6.000 genes en `X` y 36.412 en `raw`.

In [8]:
comp = guardar(data.check_expectations(adata.obs, cfg, adata.n_vars, adata.raw.n_vars), "comprobaciones")
display(comp)

por_clase = guardar(data.donors_per_class(donantes, cfg), "donantes_por_clase")
print("\nDonantes y células por clase:")
display(por_clase)

tipo_clase = guardar(pd.crosstab(adata.obs[COL["cell_type"]], adata.obs[COL["label"]])
                     [[c for c in cfg["labels"]["all"] if c in set(adata.obs[COL["label"]])]], "celulas_tipo_clase")
print("\nCélulas por tipo celular y clase:")
display(tipo_clase)

,comprobacion,esperado,observado,ok
0,Células totales,24000,24000,sí
1,Donantes,80,80,sí
2,Tipos celulares,6,6,sí
3,Células Reference,6000,6000,sí
4,Células Low,6000,6000,sí
5,Células Intermediate,6000,6000,sí
6,Células High,6000,6000,sí
7,Genes en X,6000,6000,sí
8,Genes en raw,36412,36412,sí
9,Donantes con más de un ADNC,0,0,sí



Donantes y células por clase:


,donantes,celulas,celulas_por_donante_mediana,celulas_por_donante_min,celulas_por_donante_max
ADNC,,,,,
Reference,5,6000,1242.0,143,2682
Low,12,6000,515.5,266,756
Intermediate,21,6000,271.0,153,501
High,42,6000,135.0,43,275



Células por tipo celular y clase:


ADNC,Reference,Low,Intermediate,High
cell_type_abbr,,,,
OLG,1000,1000,1000,1000
MGC,1000,1000,1000,1000
AST,1000,1000,1000,1000
VIP-GABA,1000,1000,1000,1000
SST-GABA,1000,1000,1000,1000
L2IT-GLU,1000,1000,1000,1000


### Células por donante y tipo celular

Para el pseudobulk sumaremos las células de cada (donante, tipo). Si un par tiene 0 o muy pocas células, ese
pseudobulk será ruidoso o no existirá.

In [9]:
dt_ct = pd.crosstab(adata.obs[COL["donor"]], adata.obs[COL["cell_type"]])
guardar(dt_ct, "celulas_donante_tipo")
lab = donantes[COL["label"]]
resumen_pares = pd.DataFrame({
    "pares_con_0": (dt_ct == 0).groupby(lab).sum().sum(axis=1),
    "pares_con_<10": ((dt_ct > 0) & (dt_ct < 10)).groupby(lab).sum().sum(axis=1),
    "mediana": dt_ct.groupby(lab).apply(lambda t: float(t.stack().median())),
    "min": dt_ct.groupby(lab).min().min(axis=1),
    "max": dt_ct.groupby(lab).max().max(axis=1),
}).reindex([c for c in cfg["labels"]["all"] if c in set(lab)])
guardar(resumen_pares, "pares_donante_tipo_por_clase")
display(resumen_pares)

,pares_con_0,pares_con_<10,mediana,min,max
ADNC,,,,,
Reference,4,2,196.5,0,601
Low,0,1,74.0,9,218
Intermediate,0,2,42.0,2,155
High,4,40,21.0,0,71


## 7. Sin Reference: lo que queda para clasificar

Los Reference son 5 donantes jóvenes de otra cohorte y quedan fuera de la clasificación principal. Miramos
los confusores **por donante** en las tres clases restantes: si una variable se reparte muy distinto entre
clases, el modelo podría aprender esa variable en lugar de la enfermedad.

In [10]:
ref = donantes[donantes[COL["label"]].isin(cfg["labels"]["exclude"])]
print(f"Donantes excluidos ({len(ref)}):")
display(ref[[c for c in ["n_celulas", "Age at death", "sex", "APOE4 status", COL["reference_flag"],
                         "Cognitive status"] if c in ref.columns]])

conf = data.confounder_tables(donantes, cfg, ["Age at death", "development_stage", "sex", "APOE4 status", "PMI", "Cognitive status",
                                              "Lewy body disease pathology", "LATE-NC stage"])
for c, t in conf.items():
    guardar(t, "confusor_" + c.replace(" ", "_").replace("-", "_"))
    print(f"\n--- {c} (nº de donantes) ---")
    display(t)

Donantes excluidos (5):


,n_celulas,Age at death,sex,APOE4 status,Neurotypical reference,Cognitive status
donor_id,,,,,,
H18.30.001,441,Less than 65 years old,female,Reference,True,Reference
H18.30.002,1242,Less than 65 years old,male,Reference,True,Reference
H19.30.001,2682,Less than 65 years old,male,Reference,True,Reference
H19.30.002,1492,Less than 65 years old,male,Reference,True,Reference
H200.1023,143,Less than 65 years old,female,Reference,True,Reference



--- Age at death (nº de donantes) ---


Age at death,65 to 77 years old,78 to 89 years old,90+ years old
ADNC,,,
Low,2,5,5
Intermediate,0,9,12
High,6,12,24



--- development_stage (nº de donantes) ---


development_stage,65-year-old stage,68-year-old stage,69-year-old stage,70-year-old stage,72-year-old stage,75-year-old stage,77-year-old stage,80 year-old and over stage,80-year-old stage,81-year-old stage,82-year-old stage,83-year-old stage,84-year-old stage,85-year-old stage,86-year-old stage,87-year-old stage,88-year-old stage,89-year-old stage
ADNC,,,,,,,,,,,,,,,,,,
Low,0,0,0,0,1,1,0,5,1,0,1,0,1,0,0,0,1,1
Intermediate,0,0,0,0,0,0,0,12,0,0,2,0,0,2,0,1,3,1
High,1,1,1,1,0,1,1,24,0,2,1,2,1,0,3,1,1,1


... (8 outputs left out)


--- Lewy body disease pathology (nº de donantes) ---


Lewy body disease pathology,Amygdala-predominant,Brainstem-predominant,Limbic (Transitional),Neocortical (Diffuse),Not Identified (olfactory bulb assessed),Not Identified (olfactory bulb not assessed),Olfactory bulb only
ADNC,,,,,,,
Low,0,0,2,1,3,6,0
Intermediate,0,2,1,5,1,11,1
High,10,0,4,7,4,16,1



--- LATE-NC stage (nº de donantes) ---


LATE-NC stage,LATE Stage 1,LATE Stage 2,LATE Stage 3,Not Identified,Staging Precluded by FTLD with TDP43 or ALS/MND or TDP-43 pathology is unclassifiable
ADNC,,,,,
Low,2,4,0,6,0
Intermediate,5,5,2,9,0
High,6,24,3,8,1


In [11]:
resumen = {
    "celulas": int(adata.n_obs), "genes_X": int(adata.n_vars), "genes_raw": int(adata.raw.n_vars),
    "donantes": int(len(donantes)),
    "donantes_por_clase": {k: int(v) for k, v in por_clase["donantes"].items()},
    "celulas_por_clase": {k: int(v) for k, v in por_clase["celulas"].items()},
    "tipos_celulares": sorted(map(str, adata.obs[COL["cell_type"]].unique())),
    "layers": [k for k in adata.layers.keys() if k is not None],
    "obsm": {k: list(adata.obsm[k].shape) for k in adata.obsm.keys()},
    "diagnostico_matrices": mat["diagnostico"].to_dict(),
    "normalizacion": norm["conclusion"],
    "columnas_sin_clasificar": cols_obs.loc[cols_obs.grupo == "sin_clasificar", "columna"].tolist(),
    "columnas_donante_no_constantes": [p["columna"] for p in donantes.attrs["no_constantes"]],
}
(TABLAS / "resumen.json").write_text(json.dumps(resumen, indent=2, ensure_ascii=False), encoding="utf-8")
adata.file.close()
print(json.dumps(resumen, indent=2, ensure_ascii=False))
print("\nTablas guardadas en", TABLAS)

{
  "celulas": 24000,
  "genes_X": 6000,
  "genes_raw": 36412,
  "donantes": 80,
  "donantes_por_clase": {
    "Reference": 5,
    "Low": 12,
    "Intermediate": 21,
    "High": 42
  },
  "celulas_por_clase": {
    "Reference": 6000,
    "Low": 6000,
    "Intermediate": 6000,
    "High": 6000
  },
  "tipos_celulares": [
    "AST",
    "L2IT-GLU",
    "MGC",
    "OLG",
    "SST-GABA",
    "VIP-GABA"
  ],
  "layers": [],
  "obsm": {
    "X_scVI": [
      24000,
      20
    ],
    "X_umap": [
      24000,
      2
    ]
  },
  "diagnostico_matrices": {
    "X": "log-normalizado (decimales, máx < 20); la suma no es constante porque no están todos los genes",
    "raw/X": "recuentos crudos (enteros)"
  },
  "normalizacion": "X = log1p(raw / total_celula x 9992), con el total sobre todos los genes de raw",
  "columnas_sin_clasificar": [],
  "columnas_donante_no_constantes": []
}

Tablas guardadas en C:\Users\ori\MEDAL\Code\results\tables
